# Customer Churn Prediction — Exploratory Data Analysis

## Business Problem
Customer churn refers to the loss of clients or subscribers who stop using a company's service.
In telecom, banking, or SaaS industries, acquiring a new customer costs 5–25x more than retaining an existing one.
By predicting which customers are likely to churn, businesses can proactively offer retention incentives,
reducing revenue loss and improving customer lifetime value (CLV).

**Goal:** Build a classification model that predicts churn (Yes/No) based on customer behaviour and demographics.

**Dataset:** Telco Customer Churn from Kaggle
- URL: https://www.kaggle.com/datasets/blastchar/telco-customer-churn
- Download `WA_Fn-UseC_-Telco-Customer-Churn.csv` and save to `../data/churn.csv`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 5)

df = pd.read_csv('C:\\Users\\hp\\Documents\\Projects\\Customer Churn Prediction\\data\\WA_Fn-UseC_-Telco-Customer-Churn.csv')
print('Shape:', df.shape)
df.head()

## 1. Basic Overview

In [ ]:
print('Data Types:\n', df.dtypes)
print('\nNull Values:\n', df.isnull().sum())
print('\nDescriptive Stats:\n')
df.describe()

In [ ]:
# Fix TotalCharges — stored as object with whitespace entries
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'].fillna(df['TotalCharges'].median(), inplace=True)
print('Nulls after fix:', df['TotalCharges'].isnull().sum())

## 2. Target Variable Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
churn_counts = df['Churn'].value_counts()

axes[0].pie(churn_counts, labels=churn_counts.index, autopct='%1.1f%%',
            colors=['#4C9BE8', '#E8714C'], startangle=90)
axes[0].set_title('Churn Distribution')

sns.countplot(x='Churn', data=df, ax=axes[1], palette=['#4C9BE8', '#E8714C'])
axes[1].set_title('Churn Count')
for p in axes[1].patches:
    axes[1].annotate(f'{int(p.get_height())}', (p.get_x() + 0.35, p.get_height() + 50))

plt.tight_layout()
plt.show()
print('Imbalance ratio (No:Yes):', round(churn_counts['No'] / churn_counts['Yes'], 2))

## 3. Numerical Feature Analysis

In [ ]:
num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for i, col in enumerate(num_cols):
    for churn_val, color in zip(['Yes', 'No'], ['#E8714C', '#4C9BE8']):
        df[df['Churn'] == churn_val][col].plot(kind='kde', ax=axes[i],
                                                label=f'Churn={churn_val}', color=color)
    axes[i].set_title(f'{col} Distribution by Churn')
    axes[i].legend()

plt.tight_layout()
plt.show()

## 4. Categorical Feature Analysis — Churn Rate by Category

In [ ]:
cat_cols = ['Contract', 'InternetService', 'PaymentMethod', 'gender', 'SeniorCitizen', 'TechSupport']
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    churn_rate = df.groupby(col)['Churn'].apply(lambda x: (x == 'Yes').mean() * 100)
    churn_rate.sort_values(ascending=False).plot(
        kind='bar', ax=axes[i], color='#E8714C', edgecolor='white')
    axes[i].set_title(f'Churn Rate by {col} (%)')
    axes[i].set_ylabel('Churn %')
    axes[i].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

## 5. Tenure Segmentation

In [ ]:
df['tenure_group'] = pd.cut(df['tenure'], bins=[0, 12, 24, 48, 72],
                             labels=['0-12 mo', '12-24 mo', '24-48 mo', '48-72 mo'])

tenure_churn = df.groupby('tenure_group')['Churn'].apply(lambda x: (x == 'Yes').mean() * 100)

plt.figure(figsize=(8, 4))
tenure_churn.plot(kind='bar', color='#E8714C', edgecolor='white')
plt.title('Churn Rate by Tenure Group (%)')
plt.ylabel('Churn %')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 6. Key EDA Findings

- ~26% churn rate — significant class imbalance, must use class_weight or SMOTE
- Month-to-month contract customers churn far more than 1-year or 2-year contracts
- Fiber Optic internet users churn more than DSL users
- Higher monthly charges strongly correlate with churn
- Electronic check payment method has the highest churn rate
- Customers with no tech support churn more
- First 12 months is the highest churn risk window — critical retention period
- SeniorCitizen customers churn at a higher rate